In [0]:
%python
# Create two sample DataFrames to practice join operations
from pyspark.sql import SparkSession

# DataFrame 1: Customers
customers_data = [
    (1, "Amit", "Bangalore"),
    (2, "Priya", "Mumbai"),
    (3, "Rahul", "Delhi"),
    (4, "Sneha", "Pune"),
    (5, "Vikram", "Hyderabad"),   # customer with no orders
]
customers = spark.createDataFrame(customers_data, ["cust_id", "name", "city"])

# DataFrame 2: Orders
orders_data = [
    (101, 1, "Laptop", 75000),
    (102, 1, "Mouse", 500),
    (103, 2, "Keyboard", 1500),
    (104, 3, "Monitor", 12000),
    (105, 7, "Tablet", 30000),   # order from a customer not in customers table
]
orders = spark.createDataFrame(orders_data, ["order_id", "cust_id", "product", "amount"])

display(customers)
display(orders)



In [0]:
left_anti_join = customers.join(orders, "cust_id","left_anti")
display(left_anti_join)

In [0]:
from pyspark.sql.functions import broadcast

final_result = customers.join(broadcast(orders),"cust_id","inner")
display(final_result)

In [0]:
# Handle corrupt records while reading data (CSV)
# Data with some malformed rows (bad CSV / wrong types)
corrupt_data = [
    'cust_id,name,city',
    '1,Amit,Bangalore',
    '2,Priya,Mumbai',
    '3,Rahul,Delhi',            # missing 'city' -> null, still parsed
    '4,Sneha',                  # fewer fields -> nulls, still parsed
    '5,Vikram,Bangalore,Extra', # extra field -> corrupt record
    '6,Sneha',                  # missing column
]

# Write sample CSV data to a temp location
dbutils.fs.put("/tmp/corrupt_demo/data.csv", "\n".join(corrupt_data), overwrite=True)

# PERMISSIVE mode (default): corrupt rows are kept with all fields set to null,
# and the raw malformed string is captured in the _corrupt_record column
df = (
    spark.read
    .option("mode", "PERMISSIVE")
    .option("header", "true")
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .csv("/tmp/corrupt_demo/data.csv")
)

print("--- All rows (including corrupt) ---")
display(df)

# Separate good records from corrupt ones
good_records = df.filter(df["_corrupt_record"].isNull()).drop("_corrupt_record")
corrupt_records = df.filter(df["_corrupt_record"].isNotNull()).select("_corrupt_record")

print("--- Valid records ---")
display(good_records)

print("--- Corrupt records (for quarantine / reprocessing) ---")
display(corrupt_records)

# Optionally quarantine corrupt records to a separate path for later inspection
corrupt_records.write.mode("overwrite").text("/tmp/corrupt_demo/quarantine")

# Other useful modes:
#   .option("mode", "DROPMALFORMED")  -> silently drops corrupt rows
#   .option("mode", "FAILFAST")       -> fails the job on the first corrupt row
#   .option("badRecordsPath", "/tmp/corrupt_demo/bad") -> writes bad records to a separate JSON/CSV file

print("--- All rows (including corrupt) ---")
display(df)

# Separate good records from corrupt ones
good_records = df.filter(df["_corrupt_record"].isNull()).drop("_corrupt_record")
corrupt_records = df.filter(df["_corrupt_record"].isNotNull()).select("_corrupt_record")

print("--- Valid records ---")
display(good_records)

print("--- Corrupt records (for quarantine / reprocessing) ---")
display(corrupt_records)

# Optionally quarantine corrupt records to a separate path for later inspection
corrupt_records.write.mode("overwrite").text("/tmp/corrupt_demo/quarantine")

# Other useful modes:
#   .option("mode", "DROPMALFORMED")  -> silently drops corrupt rows
#   .option("mode", "FAILFAST")       -> fails the job on the first corrupt row
#   .option("badRecordsPath", "/tmp/corrupt_demo/bad") -> writes bad records to a separate JSON/CSV file
